In [17]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [18]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

In [19]:
# Task 1 — GDP per capita vs life expectancy, Asia, 2002 vs 2007
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = px.data.gapminder()   # reload so this cell runs on its own

CONTINENT = 'Asia'
COLORS = {2002: '#A6A6A6', 2007: '#1F77B4'}   # just 2 colours: grey = before, blue = after

d = df.loc[(df['continent'] == CONTINENT) & (df['year'].isin([2002, 2007]))]
a = d[d['year'] == 2002].set_index('country')
b = d[d['year'] == 2007].set_index('country')

n_up = int((b['lifeExp'] > a['lifeExp']).sum())
avg_gain = (b['lifeExp'] - a['lifeExp']).mean()

fig = go.Figure()

# Arrows from 2002 -> 2007 show the direction of change for each country
# NOTE: annotations on a log axis need log10(x) - raw values push the chart off-screen
for c in a.index:
    fig.add_annotation(x=np.log10(b.loc[c, 'gdpPercap']), y=b.loc[c, 'lifeExp'],
                       ax=np.log10(a.loc[c, 'gdpPercap']), ay=a.loc[c, 'lifeExp'],
                       xref='x', yref='y', axref='x', ayref='y',
                       showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1,
                       arrowcolor='#CCCCCC', text='')

for year, g in d.groupby('year'):
    fig.add_trace(go.Scatter(
        x=g['gdpPercap'], y=g['lifeExp'], mode='markers', name=str(year),
        marker=dict(color=COLORS[year], size=9, line=dict(width=0)),
        text=g['country'],
        hovertemplate='%{text}<br>GDP/capita: $%{x:,.0f}<br>Life exp: %{y:.1f}<extra></extra>'))

fig.update_layout(
    title=dict(text=f"<b>{n_up} of {len(a)} countries in {CONTINENT} gained life expectancy between 2002 and 2007, "
                    f"by {avg_gain:.1f} years on average</b>",
               x=0, xanchor='left', font=dict(size=18)),
    template='simple_white', width=900, height=550,
    margin=dict(l=60, r=40, t=90, b=60),
    xaxis=dict(type='log', title='GDP per capita (US$, log scale)', showgrid=False),
    yaxis=dict(title='Life expectancy (years)', showgrid=True, gridcolor='#F2F2F2'),
    legend=dict(title=None, orientation='h', x=0, y=1.06))
fig.show()


In [20]:
# Task 2 — Bubble chart, all countries 2007, one continent highlighted
HIGHLIGHT = 'Africa'
HL_COLOR = '#D62728'
GREY = '#D9D9D9'

d = df[df['year'] == 2007].copy()
d['group'] = d['continent'].where(d['continent'] == HIGHLIGHT, 'Other continents')

fig = px.scatter(
    d, x='gdpPercap', y='lifeExp', size='pop', color='group',
    color_discrete_map={HIGHLIGHT: HL_COLOR, 'Other continents': GREY},
    category_orders={'group': ['Other continents', HIGHLIGHT]},   # draw highlight on top
    hover_name='country', size_max=50, log_x=True,
    labels={'gdpPercap': 'GDP per capita (US$, log scale)',
            'lifeExp': 'Life expectancy (years)', 'group': ''})
fig.update_traces(marker=dict(line=dict(width=0.5, color='white'), opacity=0.85))

hl = d[d['continent'] == HIGHLIGHT]
avg_hl = hl['lifeExp'].mean()
avg_rest = d[d['continent'] != HIGHLIGHT]['lifeExp'].mean()
low = hl.nsmallest(1, 'lifeExp').iloc[0]

fig.add_annotation(x=hl['gdpPercap'].median(), y=hl['lifeExp'].min(),
                   text=f"<b>{HIGHLIGHT}</b>: average life expectancy {avg_hl:.0f} years<br>"
                        f"vs {avg_rest:.0f} for the rest of the world",
                   showarrow=True, arrowhead=2, arrowcolor=HL_COLOR, ax=120, ay=60,
                   font=dict(color=HL_COLOR, size=13), align='left')
fig.add_annotation(x=low['gdpPercap'], y=low['lifeExp'], text=low['country'],
                   showarrow=False, yshift=-22, font=dict(color=HL_COLOR, size=11))

fig.update_layout(
    title=dict(text=f"<b>{HIGHLIGHT} has the lowest life expectancy of any continent: "
                    f"{avg_hl:.0f} years vs {avg_rest:.0f} for the rest of the world</b>",
               x=0, xanchor='left', font=dict(size=18)),
    template='simple_white', width=900, height=580,
    margin=dict(l=60, r=40, t=90, b=60), showlegend=False,
    xaxis=dict(showgrid=False), yaxis=dict(showgrid=True, gridcolor='#F2F2F2'))
fig.show()
